# 01 · Regresión logística desde cero: de la recta a la sigmoide

**Módulo 4 · Sesión 9** — Clasificación

## Objetivos

`01-descenso-gradiente-intuicion.ipynb` (módulo 3) ajustó una regresión múltiple por
descenso del gradiente, en forma vectorizada, sobre `rendimiento-estudiantes.csv`. Este
notebook **parte de ahí** y cambia una sola cosa: el objetivo ya no es `nota_final`
(continua) sino `aprobo` (0/1). Ese cambio obliga a:

1. Ver en código por qué la regresión lineal es un mal clasificador (predice
   probabilidades fuera de $[0, 1]$) y qué arregla la **sigmoide**.
2. Reemplazar el MSE por la **entropía cruzada**, y comprobar que el gradiente resultante
   tiene *exactamente* la misma forma que el de la regresión: $\mathbf{X}^\top(\hat{p} - y)/n$.
3. Reutilizar el descenso batch del módulo 3 sin cambiar una línea del bucle, y validar el
   resultado contra `LogisticRegression` de `scikit-learn`.
4. Interpretar los coeficientes como **razones de momios** (*odds ratios*).
5. Extender a más de dos clases con **softmax**, y ver qué pasa con la máxima
   verosimilitud cuando los datos son perfectamente separables.

No se repite qué es un gradiente ni cómo funciona el descenso: ver
`02-descenso-gradiente.md` y el notebook 01 del módulo 3.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.linear_model import LinearRegression, LogisticRegression

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los mismos datos del módulo 3, con el objetivo binario

Mismos seis predictores del notebook 01 del módulo 3. El objetivo ahora es `aprobo`
(1 si `nota_final` $\geq 3.0$). Como el dataset es sintético, sabemos que `nota_final` es una
combinación lineal de los predictores más ruido gaussiano: `aprobo` es, por tanto, un
**umbral sobre una variable latente**. Eso es exactamente el tipo de proceso para el que la
regresión logística es una buena aproximación.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")

columnas_x = [
    "promedio_anterior",
    "horas_estudio_semana",
    "asistencia_pct",
    "edad",
    "estrato",
    "trabaja",
]
X_crudo = datos[columnas_x].to_numpy(dtype=float)
y = datos["aprobo"].to_numpy(dtype=float)

print(f"X: {X_crudo.shape}   proporción de aprobados: {y.mean():.3f}")

# Estandarizar antes de descender, igual que en el módulo 3.
medias = X_crudo.mean(axis=0)
desvios = X_crudo.std(axis=0)
X_z = (X_crudo - medias) / desvios


def anadir_intercepto(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])


X_disenio = anadir_intercepto(X_z)

## 2. Por qué la recta no sirve para clasificar

Lo primero que se le ocurre a cualquiera: ajustar una regresión lineal a `aprobo` como si
fuera un número, y leer la predicción como "probabilidad de aprobar". Con un solo predictor
se ve el problema de inmediato.

In [ ]:
x1 = datos[["promedio_anterior"]].to_numpy()
lineal = LinearRegression().fit(x1, y)

malla = np.linspace(1.8, 5.2, 200).reshape(-1, 1)
pred_lineal = lineal.predict(malla)

print(f"Predicción 'lineal' para promedio_anterior = 2.0: {lineal.predict([[2.0]])[0]:.2f}")
print(f"Predicción 'lineal' para promedio_anterior = 5.0: {lineal.predict([[5.0]])[0]:.2f}")
print(f"Fracción de estudiantes con predicción fuera de [0, 1]: "
      f"{np.mean((lineal.predict(x1) < 0) | (lineal.predict(x1) > 1)):.3f}")

La recta predice "probabilidades" mayores que 1 para los promedios altos, y no hay nada en
el modelo que lo impida: una recta no está acotada. Además, el MSE castiga igual un error
de 0.3 cerca de la frontera que uno lejos de ella, donde el modelo ya acertó de sobra.

### La sigmoide

La solución es pasar la combinación lineal $z = \mathbf{x}^\top\boldsymbol{\beta}$ por una
función que la aplaste al intervalo $(0, 1)$:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

La sigmoide no es una elección arbitraria: es la inversa del **logit**. Si $p$ es la
probabilidad de aprobar, los *momios* (odds) son $p/(1-p)$, y el modelo logístico afirma que
el **logaritmo de los momios es lineal** en los predictores:

$$
\log\frac{p}{1-p} = \mathbf{x}^\top\boldsymbol{\beta}
\quad\Longleftrightarrow\quad
p = \sigma(\mathbf{x}^\top\boldsymbol{\beta})
$$

In [ ]:
def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


z = np.linspace(-8, 8, 300)
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(z, sigmoide(z))
ejes[0].axhline(0.5, color="gray", ls="--", lw=0.8)
ejes[0].axvline(0, color="gray", ls="--", lw=0.8)
ejes[0].set_title(r"$\sigma(z)$: de $(-\infty, \infty)$ a $(0, 1)$")
ejes[0].set_xlabel("z = log-momios")
ejes[0].set_ylabel("p")

p = np.linspace(0.001, 0.999, 300)
ejes[1].plot(p, np.log(p / (1 - p)))
ejes[1].set_title("logit(p) = log(p / (1 − p)): la inversa")
ejes[1].set_xlabel("p")
ejes[1].set_ylabel("log-momios")
plt.tight_layout()
plt.show()

## 3. La función de costo: entropía cruzada, no MSE

Podríamos intentar minimizar el MSE entre $\sigma(\mathbf{x}^\top\boldsymbol{\beta})$ e $y$.
Se puede, pero es mala idea, y la razón se ve mejor en una gráfica que en una demostración:
con la sigmoide en medio, el MSE **deja de ser convexo**. La alternativa correcta sale de la
**máxima verosimilitud**: si cada $y_i$ es una Bernoulli con probabilidad $p_i$, la
log-verosimilitud es

$$
\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \left[ y_i \log p_i + (1-y_i)\log(1-p_i) \right]
$$

y **minimizar su negativo** (dividido entre $n$) es minimizar la **entropía cruzada**
binaria (*log loss*):

$$
\mathcal{L}(\boldsymbol{\beta}) = -\frac{1}{n}\sum_{i=1}^n \left[ y_i \log p_i +
(1-y_i)\log(1-p_i) \right], \qquad p_i = \sigma(\mathbf{x}_i^\top\boldsymbol{\beta})
$$

Para verlo, tomamos el modelo con un solo predictor (`promedio_anterior` estandarizado),
fijamos el intercepto en su valor óptimo y recorremos el coeficiente de la pendiente.

In [ ]:
def costo_entropia(beta, X, y):
    p = np.clip(sigmoide(X @ beta), 1e-12, 1 - 1e-12)  # evita log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def costo_mse_sigmoide(beta, X, y):
    return np.mean((y - sigmoide(X @ beta)) ** 2)


X_1d = anadir_intercepto(X_z[:, [0]])
ajuste_1d = LogisticRegression(penalty=None).fit(X_z[:, [0]], y)
b0_opt = ajuste_1d.intercept_[0]

pendientes = np.linspace(-15, 25, 400)
ce = [costo_entropia(np.array([b0_opt, b1]), X_1d, y) for b1 in pendientes]
mse = [costo_mse_sigmoide(np.array([b0_opt, b1]), X_1d, y) for b1 in pendientes]

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(pendientes, ce)
ejes[0].set_title("Entropía cruzada: un solo valle")
ejes[0].set_xlabel(r"$\beta_1$ (pendiente)")
ejes[1].plot(pendientes, mse, color="C3")
ejes[1].set_title("MSE tras la sigmoide: se aplana lejos del óptimo")
ejes[1].set_xlabel(r"$\beta_1$ (pendiente)")
plt.tight_layout()
plt.show()

La entropía cruzada crece sin parar a ambos lados del óptimo, así que el gradiente siempre
apunta hacia él. El MSE con sigmoide se **aplana** en las colas: lejos del óptimo el
gradiente es casi cero, y un descenso que arranque allí puede quedarse quieto. Con varias
variables esas mesetas se combinan en mínimos locales de verdad.

### El gradiente

La derivada de $\sigma$ es $\sigma(z)(1-\sigma(z))$, y al combinarla con la entropía
cruzada por la regla de la cadena, los términos se cancelan de forma sorprendentemente
limpia:

$$
\nabla_{\boldsymbol{\beta}} \mathcal{L} = \frac{1}{n}\mathbf{X}^\top(\hat{\mathbf{p}} - \mathbf{y}),
\qquad \hat{\mathbf{p}} = \sigma(\mathbf{X}\boldsymbol{\beta})
$$

Compárese con el gradiente del módulo 3, $-\frac{2}{n}\mathbf{X}^\top(\mathbf{y} -
\mathbf{X}\boldsymbol{\beta})$: es la **misma estructura** —la matriz de diseño transpuesta
por el residual— con la predicción $\mathbf{X}\boldsymbol{\beta}$ reemplazada por
$\sigma(\mathbf{X}\boldsymbol{\beta})$ y sin el factor 2. Este patrón reaparecerá en la
sesión 11: el gradient boosting para clasificación ajusta árboles a exactamente este
residual, $y - \hat{p}$.

In [ ]:
def gradiente_entropia(beta, X, y):
    return X.T @ (sigmoide(X @ beta) - y) / len(y)


# Comprobación numérica del gradiente en un punto cualquiera, por diferencias finitas.
beta_prueba = rng.normal(size=X_disenio.shape[1]) * 0.5
grad_analitico = gradiente_entropia(beta_prueba, X_disenio, y)
grad_numerico = np.zeros_like(beta_prueba)
h = 1e-6
for j in range(len(beta_prueba)):
    e = np.zeros_like(beta_prueba)
    e[j] = h
    grad_numerico[j] = (
        costo_entropia(beta_prueba + e, X_disenio, y) - costo_entropia(beta_prueba - e, X_disenio, y)
    ) / (2 * h)

print("Máxima discrepancia analítico vs. numérico:", np.abs(grad_analitico - grad_numerico).max())

## 4. El mismo descenso batch del módulo 3

La función `descenso_batch` es idéntica a la del notebook 01 del módulo 3; solo recibe
otras funciones de costo y gradiente. **Eso es todo lo que cambia** al pasar de regresión a
clasificación.

In [ ]:
def descenso_batch(X, y, costo, gradiente, tasa, pasos):
    beta = np.zeros(X.shape[1])
    historial = [costo(beta, X, y)]
    for _ in range(pasos):
        beta = beta - tasa * gradiente(beta, X, y)
        historial.append(costo(beta, X, y))
    return beta, np.array(historial)


beta_gd, historial = descenso_batch(
    X_disenio, y, costo_entropia, gradiente_entropia, tasa=0.5, pasos=3000
)

plt.figure(figsize=(6.5, 4))
plt.plot(historial)
plt.xlabel("Paso")
plt.ylabel("Entropía cruzada")
plt.title("Convergencia del descenso batch")
plt.show()
print(f"Costo final: {historial[-1]:.5f}")

### Validación contra `scikit-learn`

`LogisticRegression` regulariza con $L_2$ **por defecto** (`C=1.0`); para comparar con el
descenso puro hay que desactivarlo con `penalty=None`. Es una trampa frecuente: se comparan
coeficientes "logísticos" que en realidad están encogidos por una penalización que nadie
pidió.

In [ ]:
sk = LogisticRegression(penalty=None, max_iter=5000).fit(X_z, y)
beta_sk = np.concatenate([sk.intercept_, sk.coef_[0]])

comparacion = pd.DataFrame(
    {"descenso (a mano)": beta_gd, "scikit-learn": beta_sk},
    index=["intercepto"] + columnas_x,
)
print(comparacion.round(4))
print(f"\nMáxima diferencia absoluta: {np.abs(beta_gd - beta_sk).max():.5f}")

## 5. Interpretar: razones de momios

En regresión lineal, $\beta_j$ era "cuánto cambia $\hat{y}$ por unidad de $x_j$". Aquí
$\beta_j$ es cuánto cambia el **log-momio**, que no es intuitivo. Lo que sí se interpreta es
$e^{\beta_j}$, la **razón de momios** (*odds ratio*): el factor por el que se multiplican los
momios de aprobar cuando $x_j$ sube una unidad (aquí, una desviación estándar, porque
estandarizamos).

In [ ]:
razones = pd.DataFrame(
    {
        "beta": beta_gd[1:],
        "odds ratio = exp(beta)": np.exp(beta_gd[1:]),
        "desv. estándar original": desvios,
    },
    index=columnas_x,
)
print(razones.round(3))

Subir el `promedio_anterior` una desviación estándar (0.48 puntos) multiplica los momios de
aprobar por 5.7; una desviación más de horas de estudio (5.2 h/semana), por 4.1; trabajar
los reduce a un 72 %. `edad` y `estrato` tienen razones cercanas a 1: casi no mueven los
momios, lo que es coherente con el proceso generador (no participan en `nota_final`).
Fíjese en que un odds ratio es **multiplicativo**: el efecto de dos desviaciones estándar no
es el doble sino el cuadrado — 5.7² ≈ 32 veces los momios.

### La frontera de decisión es una recta

Con umbral 0.5, el modelo predice "aprueba" cuando $\sigma(z) \geq 0.5$, es decir cuando
$z \geq 0$: la frontera es el **hiperplano** $\mathbf{x}^\top\boldsymbol{\beta} = 0$. En dos
dimensiones se ve como una recta — y las probabilidades cambian de forma suave a través de
ella.

In [ ]:
X_2d = X_z[:, :2]  # promedio_anterior, horas_estudio_semana (estandarizadas)
beta_2d, _ = descenso_batch(
    anadir_intercepto(X_2d), y, costo_entropia, gradiente_entropia, tasa=0.5, pasos=3000
)

g1, g2 = np.meshgrid(np.linspace(-3.5, 3, 200), np.linspace(-2, 3.5, 200))
malla_2d = anadir_intercepto(np.c_[g1.ravel(), g2.ravel()])
prob = sigmoide(malla_2d @ beta_2d).reshape(g1.shape)

plt.figure(figsize=(7, 5))
plt.contourf(g1, g2, prob, levels=20, cmap="RdBu", alpha=0.7)
plt.colorbar(label="P(aprueba)")
plt.contour(g1, g2, prob, levels=[0.5], colors="k")
plt.scatter(X_2d[y == 0, 0], X_2d[y == 0, 1], c="C3", s=14, label="no aprobó")
plt.scatter(X_2d[y == 1, 0], X_2d[y == 1, 1], c="C0", s=14, label="aprobó")
plt.xlabel("promedio_anterior (z)")
plt.ylabel("horas_estudio_semana (z)")
plt.title("Frontera de decisión (p = 0.5) y probabilidades")
plt.legend(loc="lower right")
plt.show()

## 6. Más de dos clases: softmax

Con $K$ clases, en vez de un vector $\boldsymbol{\beta}$ hay una matriz
$\mathbf{B} \in \mathbb{R}^{(p+1)\times K}$, y la sigmoide se generaliza a la **softmax**:

$$
P(y = k \mid \mathbf{x}) = \frac{e^{\mathbf{x}^\top\boldsymbol{\beta}_k}}
{\sum_{j=1}^K e^{\mathbf{x}^\top\boldsymbol{\beta}_j}}
$$

La entropía cruzada multiclase es $-\frac{1}{n}\sum_i \log P(y=y_i \mid \mathbf{x}_i)$, y el
gradiente vuelve a ser $\frac{1}{n}\mathbf{X}^\top(\hat{\mathbf{P}} - \mathbf{Y})$, con
$\mathbf{Y}$ la matriz *one-hot* de las etiquetas. Mismo patrón por tercera vez.

Lo probamos sobre tres grupos sintéticos, y de nuevo validamos contra `scikit-learn`.

In [ ]:
X_multi, y_multi = make_blobs(
    n_samples=450, centers=[[-2, 0], [2, 1], [0, 3.5]], cluster_std=1.1, random_state=SEMILLA
)
K = 3
Y_onehot = np.eye(K)[y_multi]
X_m = anadir_intercepto((X_multi - X_multi.mean(0)) / X_multi.std(0))


def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)  # estabilidad numérica: no cambia el resultado
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


def costo_softmax(B, X, Y):
    P = softmax(X @ B)
    return -np.mean(np.sum(Y * np.log(P + 1e-12), axis=1))


def gradiente_softmax(B, X, Y):
    return X.T @ (softmax(X @ B) - Y) / X.shape[0]


B = np.zeros((X_m.shape[1], K))
hist_soft = [costo_softmax(B, X_m, Y_onehot)]
for _ in range(2000):
    B = B - 0.5 * gradiente_softmax(B, X_m, Y_onehot)
    hist_soft.append(costo_softmax(B, X_m, Y_onehot))

pred_mano = softmax(X_m @ B).argmax(axis=1)
sk_multi = LogisticRegression(penalty=None, max_iter=5000).fit(X_m[:, 1:], y_multi)
pred_sk = sk_multi.predict(X_m[:, 1:])

print(f"Costo final softmax a mano: {hist_soft[-1]:.4f}")
print(f"Accuracy a mano: {np.mean(pred_mano == y_multi):.3f}   scikit-learn: {np.mean(pred_sk == y_multi):.3f}")
print(f"Predicciones que coinciden entre ambos: {np.mean(pred_mano == pred_sk):.3f}")

> Los coeficientes de softmax **no son únicos**: sumar el mismo vector a todas las columnas
> de $\mathbf{B}$ deja las probabilidades intactas (se cancela en el cociente). Por eso aquí
> se comparan predicciones y no coeficientes; `scikit-learn` resuelve la ambigüedad con una
> convención interna. Con $K=2$, softmax se reduce exactamente a la sigmoide.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-3, 3, 250), np.linspace(-3, 3, 250))
malla_m = anadir_intercepto(np.c_[g1.ravel(), g2.ravel()])
clase_malla = softmax(malla_m @ B).argmax(axis=1).reshape(g1.shape)

plt.figure(figsize=(6.5, 5.5))
plt.contourf(g1, g2, clase_malla, levels=[-0.5, 0.5, 1.5, 2.5], colors=["C0", "C1", "C2"], alpha=0.25)
Xz_multi = X_m[:, 1:]
for k, c in enumerate(["C0", "C1", "C2"]):
    plt.scatter(Xz_multi[y_multi == k, 0], Xz_multi[y_multi == k, 1], c=c, s=14, label=f"clase {k}")
plt.title("Softmax: fronteras lineales por pares de clases")
plt.legend()
plt.show()

## 7. Cuando la máxima verosimilitud no existe: separación perfecta

Hay un caso en el que el descenso del gradiente **nunca converge**, y no es un error de
implementación. Si existe una recta que separa perfectamente las dos clases, la entropía
cruzada puede hacerse tan pequeña como se quiera estirando los coeficientes: con
$\boldsymbol{\beta}$ el doble de grande, la sigmoide es el doble de empinada y cada
$p_i$ se acerca más a su etiqueta. El mínimo está "en el infinito".

Primero el argumento directo: tomamos un $oldsymbol{eta}$ que separa los datos y
evaluamos el costo de $coldsymbol{eta}$ para $c$ cada vez mayor.

In [ ]:
X_sep, y_sep = make_blobs(n_samples=100, centers=[[-2.5, 0], [2.5, 0]], cluster_std=0.6, random_state=SEMILLA)
X_sep_d = anadir_intercepto(X_sep)

beta_separa = np.array([0.0, 1.0, 0.0])  # la recta vertical x1 = 0 separa los dos grupos
print("Errores de clasificación de beta_separa:", int(np.sum((X_sep_d @ beta_separa > 0) != y_sep)))
for c in [1, 2, 4, 8, 16, 32]:
    print(f"c = {c:>2}   costo(c·beta) = {costo_entropia(c * beta_separa, X_sep_d, y_sep):.2e}")

Cada duplicación de los coeficientes baja el costo, y no hay razón para detenerse: no
existe un mínimo finito. El descenso del gradiente lo persigue — cada vez más despacio,
porque el gradiente $\mathbf{X}^	op(\hat{\mathbf{p}} - \mathbf{y})/n$ se hace diminuto
cuando todas las probabilidades ya están cerca de 0 o de 1 — pero nunca se detiene.

In [ ]:
beta_s = np.zeros(3)
normas, costos = [], []
for paso in range(20000):
    beta_s = beta_s - 0.5 * gradiente_entropia(beta_s, X_sep_d, y_sep)
    if paso % 1000 == 0:
        normas.append(np.linalg.norm(beta_s))
        costos.append(costo_entropia(beta_s, X_sep_d, y_sep))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(np.arange(len(normas)) * 1000, normas)
ejes[0].set_title(r"$\|\boldsymbol{\beta}\|$ crece sin cota")
ejes[0].set_xlabel("Paso")
ejes[1].plot(np.arange(len(costos)) * 1000, costos)
ejes[1].set_yscale("log")
ejes[1].set_title("El costo baja hacia 0, nunca llega")
ejes[1].set_xlabel("Paso")
plt.tight_layout()
plt.show()

print(f"Norma de beta tras 1 000 pasos: {normas[1]:.1f}   tras 20 000: {normas[-1]:.1f}  (sigue subiendo)")
print(f"Costo tras 20 000 pasos: {costos[-1]:.2e}")

Esto tiene una consecuencia práctica: la regularización $L_2$ que `LogisticRegression`
aplica por defecto no es solo una protección contra el sobreajuste — es lo que garantiza que
**exista una solución**. Por eso `scikit-learn` regulariza por defecto y por eso la
advertencia habitual de "*lbfgs failed to converge*" suele aparecer justo con datos
(casi) separables. Con datos reales y muchas variables, la separación perfecta es más
frecuente de lo que parece.

In [ ]:
sep_con_l2 = LogisticRegression(C=1.0, max_iter=5000).fit(X_sep, y_sep)
print(f"Norma de beta con L2 (C=1): {np.linalg.norm(np.r_[sep_con_l2.intercept_, sep_con_l2.coef_[0]]):.2f}")
print(f"Accuracy: {sep_con_l2.score(X_sep, y_sep):.3f}  (la misma: la frontera no necesita coeficientes infinitos)")

## Resumen

| Regresión (módulo 3) | Clasificación (este notebook) |
|---|---|
| $\hat{y} = \mathbf{X}\boldsymbol{\beta}$ | $\hat{p} = \sigma(\mathbf{X}\boldsymbol{\beta})$ |
| Costo: MSE | Costo: entropía cruzada (= −log-verosimilitud) |
| Gradiente: $-\frac{2}{n}\mathbf{X}^\top(\mathbf{y} - \hat{\mathbf{y}})$ | Gradiente: $\frac{1}{n}\mathbf{X}^\top(\hat{\mathbf{p}} - \mathbf{y})$ |
| $\beta_j$: cambio en $\hat{y}$ por unidad | $e^{\beta_j}$: razón de momios por unidad |
| Solución cerrada (ecuación normal) | Sin solución cerrada; y sin solución *finita* si hay separación perfecta |
| Un objetivo continuo | $K$ clases con softmax, mismo gradiente con matrices |

Lo que este notebook **no** resolvió: cómo evaluar un clasificador. La accuracy de la
sección 6 está bien para tres grupos balanceados, pero el 74 % de aprobados de la sección 1
ya avisa del problema: un modelo que dijera "todos aprueban" tendría 74 % de accuracy sin
haber aprendido nada. `03-metricas-clasificacion.md` y el notebook 02 construyen el marco
completo.